# Retail Performance Analytics

## Data Cleaning

This notebook focuses on creating a reliable analytical dataset for business analysis by addressing the data quality issues identified during the Data Understanding phase.

The goal is to create a clean and reliable analytical dataset that will be used for business analysis, SQL exploration and Power BI dashboard development.

## Project Roadmap

| Stage | Status |
|--------|--------|
| Business Understanding | ✅ Completed |
| Data Understanding | ✅ Completed |
| Data Cleaning | ✅ Completed |
| Business Analysis | 🟡 In Progress |
| SQL Analysis | ⏳ Planned |
| Power BI Dashboard | ⏳ Planned |
| Executive Report | ⏳ Planned |

## Notebook Objectives

In this notebook we will:

- Investigate the data quality issues identified during the Data Understanding phase.
- Apply business rules to clean the dataset.
- Create a reliable analytical dataset.
- Validate the cleaned dataset.
- Prepare the data for business analysis and dashboard development.

#### Import Libraries

In [80]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# Pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

# Matplotlib default style
plt.style.use("default")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12

#### Load Dataset

In [81]:
df = pd.read_excel("../data/raw/Online Retail.xlsx")

#### Initial Dataset Overview

In [82]:
rows, columns = df.shape

print(f"Rows: {rows:,}")
print(f"Columns: {columns}")

Rows: 541,909
Columns: 8


#### Observation

The cleaning process starts with the original dataset containing 541,909 retail transactions and 8 variables.

All cleaning decisions documented in this notebook will be applied to this dataset before creating the final analytical dataset.

### Data Cleaning Strategy

- The Data Understanding phase identified several data quality issues that may affect business metrics and analytical results.

- Rather than removing records indiscriminately, each issue will be investigated individually and treated according to its business meaning.

- This approach ensures that the cleaned dataset accurately represents retail sales while preserving relevant operational information whenever appropriate

## Missing Values

In [83]:
missing_values = df.isnull().sum()

missing_percentage = (
    missing_values / len(df) * 100
).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing (%)": missing_percentage
})

missing_summary = (
    missing_summary[
        missing_summary["Missing Values"] > 0
    ]
    .sort_values(
        "Missing Values",
        ascending=False
    )
)

missing_summary

,Missing Values,Missing (%)
CustomerID,135080,24.93
Description,1454,0.27


#### Observation

Only two variables contain missing values:

- CustomerID
- Description

Although both variables contain missing information, they have different business meanings and therefore require different cleaning strategies.

#### Business Impact

- Missing customer identifiers limit customer-level analyses but do not invalidate retail transactions.
- Missing product descriptions may affect product-level reporting and therefore require further investigation before defining the cleaning strategy.

#### Investigation of Missing Descriptions

Records with missing product descriptions are investigated before any cleaning action is taken.

The objective is to determine whether these observations represent valid retail transactions or internal operational activities.

In [84]:
missing_description = df[
    df["Description"].isna()
]

missing_description.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom


#### Observation

The initial sample reveals a consistent pattern.

Besides the missing product descriptions, these transactions also contain missing customer identifiers and unit prices equal to zero.

To determine whether this pattern is consistent across all observations, the remaining attributes are investigated individually.

#### Unit Price Analysis

The first verification focuses on the monetary value of these transactions.

In [85]:
missing_description["UnitPrice"].value_counts()

UnitPrice
0.0    1454
Name: count, dtype: int64

#### Observation

All records have a **UnitPrice equal to 0.00**.

This indicates that these transactions do not generate revenue and therefore differ from regular retail sales.

#### Country Analysis

The geographical distribution is examined to determine whether these records are associated with a particular market.

In [86]:
missing_description["Country"].value_counts()

Country
United Kingdom    1454
Name: count, dtype: int64

#### Observation

All transactions belong to the **United Kingdom**, indicating that this operational pattern is restricted to a single market.

#### Overall Pattern Assessment

Finally, the remaining variables are summarized to identify additional characteristics that support the cleaning decision.

In [87]:
missing_description.describe(include="all")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,1454.0,1454.0,0,1454.000000,1454,1454.0,0.0,1454
unique,1454.0,960.0,0,NaN,NaN,NaN,NaN,1
top,536414.0,35965.0,NaN,NaN,NaN,NaN,NaN,United Kingdom
freq,1.0,10.0,NaN,NaN,NaN,NaN,NaN,1454
mean,NaN,NaN,NaN,-9.359697,2011-05-22 18:40:24.883081,0.0,NaN,NaN
min,NaN,NaN,NaN,-3667.000000,2010-12-01 11:52:00,0.0,NaN,NaN
25%,NaN,NaN,NaN,-24.000000,2011-03-22 17:04:15,0.0,NaN,NaN
50%,NaN,NaN,NaN,-3.000000,2011-05-03 13:35:30,0.0,NaN,NaN
75%,NaN,NaN,NaN,4.000000,2011-08-04 15:42:45,0.0,NaN,NaN
max,NaN,NaN,NaN,5568.000000,2011-12-08 14:06:00,0.0,NaN,NaN


The investigation confirms a highly consistent operational pattern.

All **1,454** records:

- have missing **Description** values;
- have missing **CustomerID** values;
- have **UnitPrice equal to zero**;
- belong to the **United Kingdom**.

Additionally, the **Quantity** column contains both positive and negative values, indicating that these records do not represent standard retail sales.

This evidence strongly suggests that these transactions correspond to internal operational activities rather than genuine customer purchases.

#### Business Impact

Including these operational records in product-level analyses would introduce transactions without customer information, product descriptions or monetary value.

Consequently, revenue calculations, customer analyses and product performance metrics could be distorted.

#### Decision

Based on the evidence gathered during the investigation, records with missing product descriptions will be excluded from the analytical dataset.

The original dataset will remain unchanged to preserve data traceability throughout the project.

#### Implementation

In [88]:
# Create the analytical dataset
clean_df = df.copy()

# Remove records without product descriptions
clean_df = clean_df[
    clean_df["Description"].notna()
]

#### Validation

The cleaning result is validated to ensure that no records with missing product descriptions remain in the analytical dataset.

In [89]:
remaining_missing = clean_df["Description"].isna().sum()

assert remaining_missing == 0

print("✓ Validation passed: no missing product descriptions remain.")

✓ Validation passed: no missing product descriptions remain.


#### Observation

All records with missing product descriptions were successfully removed from the analytical dataset.

The original dataset remains unchanged, while the cleaned dataset will be used throughout the remaining cleaning process.

---

## Duplicate Records

### Initial Assessment

The next step is to investigate duplicated records identified during the Data Understanding phase.

Before removing any observations, it is important to determine whether they represent true duplicated entries or legitimate business transactions recorded multiple times.

In [90]:
duplicates = clean_df.duplicated().sum()

duplicate_percentage = (
    duplicates / len(clean_df) * 100
).round(2)

print(f"Duplicated records: {duplicates:,}")
print(f"Percentage: {duplicate_percentage:.2f}%")

Duplicated records: 5,268
Percentage: 0.97%


#### Observation

The cleaned dataset still contains duplicated records after removing transactions with missing product descriptions.

Before applying any cleaning rule, these records must be inspected to determine whether they are exact duplicates or represent legitimate business events.

#### Business Impact

Duplicated records may artificially inflate sales volume, quantities sold and revenue calculations.

However, removing legitimate repeated transactions could result in the loss of valid business information.

Therefore, duplicated records must be investigated before defining the cleaning strategy.

#### Investigation of Duplicated Records

A sample of duplicated records is examined to verify whether the duplicated observations are identical across all variables.

In [91]:
duplicate_df = clean_df[
    clean_df.duplicated(keep=False)
]

duplicate_df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom


In [92]:
duplicate_df.duplicated().value_counts()

True     5268
False    4879
Name: count, dtype: int64

In [93]:
duplicate_df.shape

(10147, 8)

In [94]:
duplicate_df.nunique()

InvoiceNo      1933
StockCode      1706
Description    1732
Quantity         35
InvoiceDate    1914
UnitPrice        80
CustomerID      960
Country          14
dtype: int64

#### Observation

The investigation confirms that the duplicated observations are exact duplicates across all available variables.

The sample inspection shows identical values for invoice number, product, quantity, transaction date, customer, price and country, indicating that these records do not represent distinct business events.

#### Business Impact

Keeping exact duplicated records would artificially inflate transaction counts, quantities sold and revenue calculations.

Removing these duplicated observations improves data reliability without affecting legitimate business transactions.

#### Implementation

In [95]:
rows_before = len(clean_df)

# Remove exact duplicated transactions
clean_df = clean_df.drop_duplicates()

rows_after = len(clean_df)

print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows after cleaning: {rows_after:,}")
print(f"Duplicated records removed: {rows_before - rows_after:,}")

Rows before cleaning: 540,455
Rows after cleaning: 535,187
Duplicated records removed: 5,268


#### Validation

The cleaned dataset is validated to ensure that no duplicated records remain.

In [96]:
remaining_duplicates = clean_df.duplicated().sum()

print(f"Remaining duplicated records: {remaining_duplicates}")

Remaining duplicated records: 0


#### Observation

All exact duplicated records were successfully removed from the analytical dataset.

Each transaction is now represented by a single unique record, reducing the risk of overstating sales metrics in subsequent analyses.

---

## Negative Quantity

### Initial Assessment

Negative quantities were identified during the Data Understanding phase.

Before removing these records, it is necessary to determine whether they represent data quality issues or legitimate business events such as cancelled orders, product returns or inventory adjustments.

In [97]:
negative_quantity_df = clean_df[
    clean_df["Quantity"] < 0
]

negative_records = len(negative_quantity_df)

negative_percentage = round(
    negative_records / len(clean_df) * 100,
    2
)

print(f"Negative quantity records: {negative_records:,}")
print(f"Percentage: {negative_percentage:.2f}%")

Negative quantity records: 9,725
Percentage: 1.82%


#### Observation

The cleaned dataset still contains **9,725 transactions** with negative quantities, representing approximately **1.82%** of all records.

Although these transactions represent a small proportion of the dataset, their business meaning must be investigated before applying any cleaning rule.

#### Business Impact

Negative quantities directly affect sales volume, revenue and inventory-related metrics.

However, these transactions may represent legitimate business events such as cancelled orders, customer returns or inventory adjustments.

Removing them without investigation could result in the loss of valuable business information.

#### Investigation of Negative Quantity Transactions

A sample of negative quantity transactions is examined to better understand their characteristics before defining the cleaning strategy.

In [98]:
negative_quantity_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


#### Observation

The sample shows valid product descriptions and positive unit prices.

This indicates that negative quantities are not necessarily data entry errors.

The invoice numbering pattern will be investigated next to determine whether these transactions correspond to cancelled orders.

#### Invoice Number Investigation

In this dataset, invoice numbers starting with the letter **"C"** indicate cancelled transactions.

The following analysis measures how many negative quantity records follow this business rule.

In [99]:
invoice_pattern = (
    negative_quantity_df["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

invoice_summary = (
    invoice_pattern
    .value_counts()
    .rename_axis("Cancelled Invoice")
    .reset_index(name="Transactions")
)

invoice_summary["Percentage (%)"] = (
    invoice_summary["Transactions"]
    / invoice_summary["Transactions"].sum()
    * 100
).round(2)

invoice_summary

,Cancelled Invoice,Transactions,Percentage (%)
0,True,9251,95.13
1,False,474,4.87


#### Observation

The invoice number investigation reveals that **9,251 out of 9,725 negative quantity transactions (95.13%)** are associated with invoice numbers starting with the letter **"C"**.

This strongly indicates that most negative quantities correspond to cancelled orders or customer returns rather than data quality issues.

Only **474 transactions (4.87%)** have negative quantities without a cancelled invoice number and therefore require further investigation.

#### Business Impact

Cancelled transactions do not represent completed sales and therefore should not be included in sales performance analyses.

However, transactions with negative quantities that are not associated with cancelled invoices may represent inventory adjustments or operational corrections rather than customer purchases.

These records require additional investigation before defining the final cleaning rule.

In [100]:
non_cancelled_negative = negative_quantity_df[
    ~negative_quantity_df["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
]

non_cancelled_negative.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
7313,537032,21275,?,-30,2010-12-03 16:50:00,0.0,NaN,United Kingdom
13217,537425,84968F,check,-20,2010-12-06 15:35:00,0.0,NaN,United Kingdom
13218,537426,84968E,check,-35,2010-12-06 15:36:00,0.0,NaN,United Kingdom
13264,537432,35833G,damages,-43,2010-12-06 16:10:00,0.0,NaN,United Kingdom
21338,538072,22423,faulty,-13,2010-12-09 14:10:00,0.0,NaN,United Kingdom
21518,538090,20956,?,-723,2010-12-09 14:48:00,0.0,NaN,United Kingdom
22296,538161,46000S,Dotcom sales,-100,2010-12-09 17:25:00,0.0,NaN,United Kingdom
22297,538162,46000M,Dotcom sales,-100,2010-12-09 17:25:00,0.0,NaN,United Kingdom
42564,540010,22501,reverse 21/5/10 adjustment,-100,2011-01-04 11:13:00,0.0,NaN,United Kingdom
42566,540012,22502,reverse 21/5/10 adjustment,-100,2011-01-04 11:14:00,0.0,NaN,United Kingdom


In [101]:
non_cancelled_negative["Description"].value_counts().head(20)

Description
check                         120
damages                        45
damaged                        42
?                              41
sold as set on dotcom          20
Damaged                        14
thrown away                     9
Unsaleable, destroyed.          9
??                              7
damages?                        5
wet damaged                     5
ebay                            5
smashed                         4
missing                         3
CHECK                           3
wet pallet                      3
Dotcom sales                    2
reverse 21/5/10 adjustment      2
counted                         2
crushed                         2
Name: count, dtype: int64

#### Observation

The remaining negative quantity transactions without cancelled invoice numbers exhibit a consistent operational pattern.

Most records contain descriptions such as **"check"**, **"damages"**, **"faulty"**, **"Dotcom sales"**, and **"reverse adjustment"**.

Additionally, these transactions have:

- no customer identification;
- unit prices equal to zero;
- invoice numbers that do not represent cancelled sales.

These characteristics indicate internal operational activities rather than customer purchases.

#### Business Impact

Operational adjustments do not represent completed retail transactions.

Including these records in the analytical dataset would distort sales volume, inventory movements and revenue-related metrics.

Therefore, they should not be considered in business performance analyses.

### Decision

The investigation showed that negative quantity records represent either:

- cancelled transactions, or
- internal operational adjustments.

Neither scenario corresponds to completed retail sales.

Therefore, all transactions with negative quantities will be excluded from the analytical dataset used in this project.

Cancelled transactions may be retained in a separate dataset for future analyses such as return rate, product quality monitoring or loss prevention.

#### Implementation

The analytical dataset will retain only completed sales transactions.

Consequently:

- cancelled transactions identified by invoice numbers starting with **"C"** will be removed;
- operational adjustment records with negative quantities will also be removed.

This ensures that the analytical dataset represents only valid sales transactions.

In [102]:
rows_before = len(clean_df)

# Remove transactions that do not represent completed sales
clean_df = clean_df[
    clean_df["Quantity"] > 0
]

rows_after = len(clean_df)

print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows after cleaning: {rows_after:,}")
print(f"Records removed: {rows_before - rows_after:,}")

Rows before cleaning: 535,187
Rows after cleaning: 525,462
Records removed: 9,725


#### Validation

In [103]:
remaining_negative = (
    clean_df["Quantity"] < 0
).sum()

print(f"Remaining negative quantities: {remaining_negative}")

Remaining negative quantities: 0


#### Observation

All transactions with negative quantities were successfully removed from the analytical dataset.

The resulting dataset now contains only positive-quantity retail sales, providing a consistent foundation for business analysis and KPI development.

---

## Negative Unit Price

### Initial Assessment

Negative unit prices were identified during the Data Understanding phase.

Since negative prices are uncommon in retail sales, these records are investigated to determine whether they represent valid business transactions or operational adjustments.

In [104]:
negative_price_df = clean_df[
    clean_df["UnitPrice"] < 0
]

negative_price_records = len(negative_price_df)

negative_price_percentage = round(
    negative_price_records / len(clean_df) * 100,
    4
)

print(f"Negative unit price records: {negative_price_records}")
print(f"Percentage: {negative_price_percentage:.4f}%")

Negative unit price records: 2
Percentage: 0.0004%


In [105]:
negative_price_df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


#### Observation

The investigation identified only **2 transactions (0.0004%)** with negative unit prices.

Both records correspond to **"Adjust bad debt"** entries, contain no customer identification and represent accounting adjustments rather than retail sales transactions.

#### Business Impact

Accounting adjustments do not represent product sales and therefore should not be included in sales performance analyses.

Although these records represent a negligible proportion of the dataset, retaining them could distort revenue-related metrics.

#### Decision

Since these records correspond to accounting adjustments rather than completed retail sales, they will be removed from the analytical dataset.

#### Implementation

In [106]:
rows_before = len(clean_df)

# Remove accounting adjustments with negative unit prices
clean_df = clean_df[
    clean_df["UnitPrice"] >= 0
]

rows_after = len(clean_df)

print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows after cleaning: {rows_after:,}")
print(f"Records removed: {rows_before - rows_after:,}")

Rows before cleaning: 525,462
Rows after cleaning: 525,460
Records removed: 2


#### Validation

The cleaned dataset is validated to ensure that no records with negative unit prices remain.

In [107]:
remaining_negative_prices = (
    clean_df["UnitPrice"] < 0
).sum()

print(f"Remaining negative unit prices: {remaining_negative_prices}")

Remaining negative unit prices: 0


#### Observation

All accounting adjustment records with negative unit prices were successfully removed from the analytical dataset.

The cleaned dataset now contains only valid retail sales transactions with non-negative quantities and unit prices.

---

## Final Dataset Validation

In [108]:
print(f"Final dataset shape: {clean_df.shape}")

print(f"Missing descriptions: {clean_df['Description'].isna().sum()}")

print(f"Duplicated records: {clean_df.duplicated().sum()}")

print(f"Negative quantities: {(clean_df['Quantity'] < 0).sum()}")

print(f"Negative unit prices: {(clean_df['UnitPrice'] < 0).sum()}")

Final dataset shape: (525460, 8)
Missing descriptions: 0
Duplicated records: 0
Negative quantities: 0
Negative unit prices: 0


#### Observation

The final validation confirms that all targeted data quality issues identified during the cleaning process have been successfully addressed.

Missing customer identifiers remain in the dataset by design, as anonymous retail transactions represent valid business events and were intentionally preserved for sales analysis.
Negative unit prices: 0

## Data Cleaning Summary

The data cleaning process applied business-driven rules to prepare a reliable analytical dataset for retail sales analysis.

### Cleaning actions performed

- Removed records with missing product descriptions.
- Removed exact duplicated transactions.
- Removed transactions with negative quantities.
- Removed accounting adjustment records with negative unit prices.
- Preserved anonymous customer transactions to maintain sales completeness.

### Final Result

The resulting analytical dataset contains **525,460 valid retail transactions** ready for business analysis, SQL exploration and dashboard development.

All cleaning decisions were based on business rules identified during the investigation process rather than applying generic data cleaning techniques.

The original dataset remains unchanged, ensuring full traceability and reproducibility throughout the project.

### Export Clean Dataset

In [109]:
clean_df.to_csv(
    "../data/processed/online_retail_clean.csv",
    index=False
)

print("Clean dataset successfully exported.")

Clean dataset successfully exported.


## Next Step

The next notebook focuses on **Business Analysis**, where the cleaned dataset will be used to calculate business KPIs, explore customer purchasing behaviour and generate actionable business insights.

Although cancelled transactions were excluded from this analytical dataset, they may support future analyses related to returns, operational performance and product quality.